In [2]:
# Minimal imports (pip install if needed)
import sys, re, unicodedata, json
from pathlib import Path
import pandas as pd
import numpy as np

# If scikit-learn isn't available, uncomment:
# !pip install -q scikit-learn

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, top_k_accuracy_score

# Set your data paths (use the uploaded copies you mentioned)
UNI_PATH   = Path(r"C:\Users\zacha\OneDrive\Projects\漢字音庫\Unihan_Readings.txt")
OC_XLSX    = Path(r"C:\Users\zacha\OneDrive\Projects\漢字音庫\BaxterSagartOC2015-10-13.xlsx")
WIKIHAN    = Path(r"C:\Users\zacha\OneDrive\Projects\漢字音庫\wikihan-ipa.tsv")
HAKKA_XLSX = Path(r"C:\Users\zacha\OneDrive\Projects\漢字音庫\客家話.xlsx")
TEO_XLSX   = Path(r"C:\Users\zacha\OneDrive\Projects\漢字音庫\潮州話.xlsx")

UNI_PATH, OC_XLSX, WIKIHAN, HAKKA_XLSX, TEO_XLSX


(WindowsPath('C:/Users/zacha/OneDrive/Projects/漢字音庫/Unihan_Readings.txt'),
 WindowsPath('C:/Users/zacha/OneDrive/Projects/漢字音庫/BaxterSagartOC2015-10-13.xlsx'),
 WindowsPath('C:/Users/zacha/OneDrive/Projects/漢字音庫/wikihan-ipa.tsv'),
 WindowsPath('C:/Users/zacha/OneDrive/Projects/漢字音庫/客家話.xlsx'),
 WindowsPath('C:/Users/zacha/OneDrive/Projects/漢字音庫/潮州話.xlsx'))

In [26]:
# === Load sources & build df_raw (robust, with forced OC/MC join) ===
import pandas as pd, re, unicodedata
from pathlib import Path

# 1) Load sources (with basic diagnostics)
unihan = load_unihan_with_extras_fast(UNI_PATH)  # fast/unpivoted

try:
    oc_map, mc_map = load_baxter_sagart_excel(OC_XLSX)  # may return {} if column-pick fails
except Exception as e:
    print("Warn: load_baxter_sagart_excel failed:", e)
    oc_map, mc_map = {}, {}

try:
    add_map = load_additional_dialects(WIKIHAN)
except Exception as e:
    print("Warn: cannot load wikihan-ipa.tsv:", e)
    add_map = {}

try:
    hakka_map = load_hakka_from_excel(HAKKA_XLSX)
except Exception as e:
    print("Warn: cannot load Hakka:", e)
    hakka_map = {}

try:
    teo_map = load_teochew_from_excel(TEO_XLSX)
except Exception as e:
    print("Warn: cannot load Teochew:", e)
    teo_map = {}

print(f"[DIAG] dict sizes — Unihan:{len(unihan):,} OC:{len(oc_map):,} MC:{len(mc_map):,} "
      f"Wikihan:{len(add_map):,} Hakka:{len(hakka_map):,} Teochew:{len(teo_map):,}")

# 2) Build df_raw from union of all characters
all_chars = set()
for d in (unihan, oc_map, mc_map, add_map, hakka_map, teo_map):
    if isinstance(d, dict):
        all_chars |= set(d.keys())
df_raw = pd.DataFrame({"char": sorted(all_chars)})

# 3) Merge everything into df_raw (non-destructive fill)
df_raw = merge_sources_into_df(
    df_raw,
    unihan_dict=unihan,
    oc_map=oc_map,
    mc_map=mc_map,
    wiki_dict=add_map,
    hakka_map=hakka_map,
    teochew_map=teo_map,
)

# 4) Force-join OC & MC directly from the Excel (handles wonky headers/char parsing)
df_bs = pd.read_excel(OC_XLSX, dtype=object)
df_bs.columns = [str(c).strip() for c in df_bs.columns]

def _pick(colnames, patterns):
    """Pick first column whose name matches ANY regex in patterns (case-insensitive)."""
    for pat in patterns:
        rx = re.compile(pat, flags=re.IGNORECASE)
        for c in colnames:
            if rx.search(str(c)):
                return c
    return None

char_col = _pick(df_bs.columns, [r'^(zi|char|character|字|graph|hanzi|ch)\b'])
oc_col   = _pick(df_bs.columns, [r'\bOC\b', r'Old[_\s-]*Chinese', r'Baxter[–-]?\s*Sagart.*OC'])
mc_col   = _pick(df_bs.columns, [r'\bMC\b', r'Middle[_\s-]*Chinese', r'Baxter[–-]?\s*Sagart.*MC'])
print(f"[BS join] picked columns -> char: {char_col}, OC: {oc_col}, MC: {mc_col}")

HEX_RE = re.compile(r'(?:U\+)?([0-9A-Fa-f]{4,6})')
def _parse_char(cell):
    s = "" if cell is None else str(cell).strip()
    if not s:
        return ""
    s = unicodedata.normalize("NFC", s)
    if len(s) == 1:
        return s
    m = HEX_RE.search(s)
    if m:
        try:
            return chr(int(m.group(1), 16))
        except Exception:
            return ""
    for ch in s:
        if ch.strip():
            return ch
    return ""

if char_col is not None:
    df_bs['char'] = df_bs[char_col].map(_parse_char)
    keep_cols = ['char']
    if oc_col: keep_cols.append(oc_col)
    if mc_col: keep_cols.append(mc_col)
    df_bs = df_bs[keep_cols].dropna(subset=['char'])
    df_bs = df_bs[df_bs['char'].ne("")].copy()

    rename_map = {}
    if oc_col: rename_map[oc_col] = 'Old Chinese'
    if mc_col: rename_map[mc_col] = 'Middle Chinese'
    df_bs = df_bs.rename(columns=rename_map)

    # Left-join onto df_raw; only fill blanks
    df_raw = df_raw.merge(df_bs, on='char', how='left', suffixes=('', '_bs'))

    for col in ['Old Chinese', 'Middle Chinese']:
        if (col + '_bs') in df_raw.columns:
            mask = df_raw[col].astype(str).fillna("").str.strip().eq("")
            df_raw.loc[mask, col] = df_raw.loc[mask, col + '_bs']
            df_raw.drop(columns=[col + '_bs'], inplace=True)
else:
    print("[BS join] Could not find a character column in BS Excel; skipping forced join.")

# 5) Ensure expected columns exist
ALL_COLS = [
    'Definition',
    'Mandarin','Cantonese','Teochew','Hokkien','Hakka','Wu','Gan','Xiang','Jin',
    'Vietnamese','Japanese','Japanese On','Japanese Kun',
    'Korean (Hangul)','Korean (romanized)','Zhuang','Tang',
    'Middle Chinese','Fanqie','fanqie_upper','fanqie_lower',
    'Old Chinese'
]
for c in ALL_COLS:
    if c not in df_raw.columns:
        df_raw[c] = ""

# 6) Correct non-empty coverage (don’t count the string "nan")
def nonempty_count(s):
    s = s.astype(object)
    return int(((~pd.isna(s)) & (s.astype(str).str.strip().ne(""))).sum())

coverage = {k: nonempty_count(df_raw[k]) for k in ['Old Chinese','Middle Chinese','Mandarin','Cantonese','Teochew','Hokkien','Hakka','Wu','Gan','Xiang','Jin','Vietnamese']}
print("[COVERAGE]", coverage)

print("df_raw shape:", df_raw.shape)
df_raw.sample(5)


[Unihan-fast] Reading: C:\Users\zacha\OneDrive\Projects\漢字音庫\Unihan_Readings.txt
[Unihan-fast] Raw non-comment lines: 288,570
[Unihan-fast] Kept rows: 223,354
[Unihan-fast] Rows with valid char: 223,354
[Unihan-fast] Pivoting…
[Unihan-fast] Splitting Fanqie…
[Unihan-fast] Converting to dict…


Unihan chars: 100%|██████████████████████████████████████████████████████████| 67400/67400 [00:00<00:00, 629588.83ch/s]


[Unihan-fast] Done. Unique chars: 67,400
[BS] Reading Excel: C:\Users\zacha\OneDrive\Projects\漢字音庫\BaxterSagartOC2015-10-13.xlsx (sheet=None)


C:\Users\zacha\miniconda3\envs\env1\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Warn: load_baxter_sagart_excel failed: 'dict' object has no attribute 'columns'
[Wiki] Reading TSV: C:\Users\zacha\OneDrive\Projects\漢字音庫\wikihan-ipa.tsv
[Wiki] Shape: (21227, 6)
[Wiki] Usable rows: 21,227


Wiki chars: 100%|██████████████████████████████████████████████████████████████| 17321/17321 [00:09<00:00, 1737.66ch/s]


... (4 outputs left out)

[Teochew] Done. Chars: 7,477
[DIAG] dict sizes — Unihan:67,400 OC:0 MC:0 Wikihan:5,727 Hakka:4,395 Teochew:7,477


C:\Users\zacha\miniconda3\envs\env1\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


[BS join] picked columns -> char: zi, OC: OC, MC: MC
[COVERAGE] {'Old Chinese': 4966, 'Middle Chinese': 4965, 'Mandarin': 42603, 'Cantonese': 30844, 'Teochew': 8271, 'Hokkien': 6274, 'Hakka': 5072, 'Wu': 3176, 'Gan': 1425, 'Xiang': 1501, 'Jin': 1676, 'Vietnamese': 8897}
df_raw shape: (68333, 24)


,char,Definition,Mandarin,Cantonese,Tang,Japanese On,Japanese Kun,Japanese,Korean (romanized),Korean (Hangul),...,fanqie_lower,Old Chinese,Middle Chinese,Gan,Jin,Hokkien,Wu,Xiang,Hakka,Teochew
9535,壟,"grave, mound; ridge in field",lǒng,lung5,liǒng,RYOU ROU,UNE TSUKA,ロウ リョウ リュ ル うね つか,LONG,농:0 롱:0N,...,,NaN,NaN,,,li̯ɔŋ˥˩/lɔŋ˥˩/li̯əŋ˨/li̯əŋ˧,,,,long2
4394,䔦,,,mui4,,,,,,,...,,NaN,NaN,,,,,,,
41757,𤗴,,tuí,,,,,タイ デ,,,...,,NaN,NaN,,,,,,,
7399,僵,"stiff and motionless, stock still",jiāng,goeng1,giɑng,KYOU,TAORERU,キョウ コウ たおれる たおす,KANG,강:1N,...,,NaN,NaN,,,ki̯ɔŋ˥/kʰi̯ɔŋ˥/ki̯aŋ˥,,,kioŋ (陰平44),giang1
58710,𩯸,,,,,,,サ,,,...,,NaN,NaN,,,,,,,


In [27]:
df_raw.to_csv('combined_dialects.csv', index=False)

In [28]:
import re, unicodedata, numpy as np, pandas as pd

# =========================
# Unicode helpers / regexes
# =========================

# Extended CJK ranges (A..I blocks + compatibility/extension blocks)
_CJK_BLOCKS = (
    r'\u3400-\u4DBF'   # CJK Ext A
    r'\u4E00-\u9FFF'   # CJK Unified Ideographs
    r'\uF900-\uFAFF'   # CJK Compatibility Ideographs
    r'\U00020000-\U0002A6DF'  # CJK Ext B
    r'\U0002A700-\U0002B73F'  # CJK Ext C
    r'\U0002B740-\U0002B81F'  # CJK Ext D
    r'\U0002B820-\U0002CEAF'  # CJK Ext E
    r'\U0002CEB0-\U0002EBEF'  # CJK Ext F
    r'\U00030000-\U0003134F'  # CJK Ext G
    r'\U00031350-\U000323AF'  # CJK Ext H (if present)
    r'\U0002EBF0-\U0002EE5F'  # CJK Ext I (if present)
)
# One big class that matches ANY CJK char
_CJK = re.compile(f'[{_CJK_BLOCKS}]')

# Superscript digits → ASCII (¹..⁶ etc.)
_SUPER_TO_ASCII = str.maketrans({
    '¹':'1','²':'2','³':'3','⁴':'4','⁵':'5','⁶':'6','⁷':'7','⁸':'8','⁹':'9','⁰':'0'
})

# Tone letters (Chao) → digits
# Mapping single tone letter to a digit; if a string has multiple letters, we'll keep them all
_TONE_LETTER_TO_DIGIT = {
    '˥':'5', '˦':'4', '˧':'3', '˨':'2', '˩':'1'
}

# Multi-reading splitter: commas, pipes, slashes, semicolons, Chinese punctuation, whitespace around them
_MULTI_SPLIT = re.compile(r'[,\|/;、，／；、\uFF0C\uFF0F\uFF1B\u3001]+')

# Tie-bar variants to unify
_TIE_BAR_VARIANTS = [
    '\u0361',  # COMBINING DOUBLE INVERTED BREVE (preferred)
    '\u035C',  # COMBINING DOUBLE BREVE BELOW
    '͡',       # visually common
]

def _unify_tie_bar(s: str) -> str:
    # Replace any variant with U+0361
    for tb in _TIE_BAR_VARIANTS:
        if tb != '\u0361':
            s = s.replace(tb, '\u0361')
    return s

# ==================================
# Normalization & multi-reading utils
# ==================================

def norm_text(x, *, strip_cjk: bool = True) -> str:
    """
    Normalize a reading:
      - NFC
      - (optional) strip all CJK chars
      - unify tie-bar
      - collapse whitespace
      - map superscript digits to ASCII
      - keep IPA/Latin/diacritics as-is
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    s = unicodedata.normalize("NFC", str(x))
    if strip_cjk:
        s = _CJK.sub("", s)
    s = _unify_tie_bar(s)
    s = s.translate(_SUPER_TO_ASCII)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def split_multi(s: str):
    if not s:
        return []
    parts = _MULTI_SPLIT.split(s)
    return [p.strip() for p in parts if p and p.strip()]

def best_reading(s: str) -> str:
    """
    Return the first reading token (post-splitting). Keep behavior simple/explicit.
    """
    parts = split_multi(s)
    return parts[0] if parts else ""

# ===================
# Tone feature helpers
# ===================

def _map_tone_letters_to_digits(s: str) -> str:
    """Map tone letters (˥˦˧˨˩) to digits 5..1, preserving order."""
    if not s:
        return ""
    out_chars = []
    for ch in s:
        out_chars.append(_TONE_LETTER_TO_DIGIT.get(ch, ch))
    return ''.join(out_chars)

def _extract_digit_stream(s: str) -> str:
    """
    From a reading string, produce a digit-only stream suitable for feature extraction.
    - First convert superscript digits and tone letters to ASCII digits.
    - Keep digits 1..9 (most dialect tone digits are 1..6; we won’t force-cap here).
    """
    if not s:
        return ""
    s2 = _map_tone_letters_to_digits(s.translate(_SUPER_TO_ASCII))
    return ''.join(ch for ch in s2 if ch.isdigit())

def tone_digits_features(s: str) -> np.ndarray:
    """
    Compute [mean, max, length, has] from tone digit cues in a string.
    Works with:
      - raw tone digits: 1..6 (Cantonese), 1..5 (Chao)
      - superscripts: ¹..⁶
      - tone letters: ˥˦˧˨˩
    Heuristics:
      - prefer contiguous sequences of length 2–4; else fall back to all digits found
    """
    if not s:
        return np.array([0., 0., 0., 0.], dtype=float)

    # Normalize reading (do NOT strip CJK here; digits/letters are what we need)
    s_norm = norm_text(s, strip_cjk=True)
    digit_stream = _extract_digit_stream(s_norm)
    if not digit_stream:
        return np.array([0., 0., 0., 0.], dtype=float)

    # Try to find 2–4 length runs first (e.g., 35, 214, 53)
    runs = re.findall(r'(\d{2,4})', digit_stream)
    if runs:
        vals = []
        for r in runs:
            ds = [int(d) for d in r]
            vals.append([float(np.mean(ds)), float(np.max(ds)), float(len(ds)), 1.0])
        return np.mean(np.array(vals, dtype=float), axis=0)

    # Fallback: use all digits found
    ds = [int(d) for d in digit_stream]
    return np.array([float(np.mean(ds)), float(np.max(ds)), float(len(ds)), 1.0], dtype=float)

def add_tone_features_block(df: pd.DataFrame, colname: str, prefix: str) -> pd.DataFrame:
    """
    Create a 4-col numeric block for `colname`:
      {prefix}_tone_mean, {prefix}_tone_max, {prefix}_tone_len, {prefix}_tone_has
    Safe if column is missing (returns zeros).
    """
    if colname not in df.columns:
        # Return a zero block with correct length
        zeros = np.zeros((len(df), 4), dtype=float)
        return pd.DataFrame(
            zeros,
            columns=[f"{prefix}_tone_mean", f"{prefix}_tone_max", f"{prefix}_tone_len", f"{prefix}_tone_has"]
        )

    arr = np.vstack([tone_digits_features(best_reading(x)) for x in df[colname]])
    return pd.DataFrame(
        arr,
        columns=[f"{prefix}_tone_mean", f"{prefix}_tone_max", f"{prefix}_tone_len", f"{prefix}_tone_has"]
    )


In [40]:
# === Cell 6: Build ML table (X, y) for multi-dialect prediction (scoped, Tang-safe) ===
import numpy as np
import pandas as pd
import re
import unicodedata

# -----------------------------
# 0) Canonicalize column names
# -----------------------------
# Normalize and trim headers so 'Tang ', 'Tang　' etc. become 'Tang'
_df_cols_norm = df_raw.rename(columns=lambda x: unicodedata.normalize("NFKC", str(x)).strip())

# Common aliases -> canonical names
ALIASES = {
    "Tang Chinese": "Tang",
    "唐音": "Tang",
}
for old, new in ALIASES.items():
    if old in _df_cols_norm.columns and "Tang" not in _df_cols_norm.columns:
        _df_cols_norm = _df_cols_norm.rename(columns={old: new})

# Work copy
df_ml = _df_cols_norm.copy()

# ----------------------------------------
# 1) Restrict to the dialects you want ML
# ----------------------------------------
DIALECT_TEXT_COLS = [
    "Old Chinese",
    "Mandarin","Cantonese","Teochew","Hokkien","Hakka",
    "Vietnamese","Japanese On","Korean (romanized)",
    "Middle Chinese","Tang",
]

# ------------------------------------------------
# 2) Ensure required columns exist (but warn once)
# ------------------------------------------------
REQUIRED = set(DIALECT_TEXT_COLS + ["Japanese Kun","fanqie_upper","fanqie_lower","char"])
missing = [c for c in REQUIRED if c not in df_ml.columns]
if missing:
    print("WARNING (Cell 6): creating empty columns for:", missing)
for col in missing:
    df_ml[col] = ""

# -------------------------------------------
# 3) Normalize dialect text, preserving CJK
#    where appropriate (Tang + fanqie)
# -------------------------------------------
# Keep Han chars for these (else they become empty)
KEEP_CJK = {"Tang", "fanqie_upper", "fanqie_lower"}

def _safe_norm_by_col(col_name, s):
    try:
        # Keep CJK for columns that store Hanzi-based readings/tokens
        return norm_text(best_reading(s), strip_cjk=(col_name not in KEEP_CJK))
    except Exception:
        return ""

for c in DIALECT_TEXT_COLS:
    df_ml[c] = df_ml[c].fillna("").astype(str).map(lambda x, col=c: _safe_norm_by_col(col, x))

# ------------------------------------------------
# 4) Auxiliary features (presence, fanqie tokens)
# ------------------------------------------------
df_ml["has_kun"] = df_ml["Japanese Kun"].fillna("").astype(str).str.strip().ne("").astype(int)
df_ml["fanqie_upper_tok"] = df_ml["fanqie_upper"].fillna("").astype(str)
df_ml["fanqie_lower_tok"] = df_ml["fanqie_lower"].fillna("").astype(str)

# -------------------------------------------------------
# 5) Tone features (restricted to your chosen dialects)
# -------------------------------------------------------
tone_cols = [
    "Mandarin","Cantonese","Teochew","Hokkien","Hakka",
    "Middle Chinese","Vietnamese","Tang",
]
tone_blocks = [add_tone_features_block(df_ml, c, prefix=c.lower()[:4]) for c in tone_cols]
tone_df = pd.concat(tone_blocks, axis=1) if tone_blocks else pd.DataFrame(index=df_ml.index)

# ------------------------------------------------
# 6) Final feature frame (same rows as df_raw)
# ------------------------------------------------
df_feat = pd.concat(
    [
        df_ml[["char"] + DIALECT_TEXT_COLS + ["fanqie_upper_tok","fanqie_lower_tok","has_kun"]].reset_index(drop=True),
        tone_df.reset_index(drop=True),
    ],
    axis=1,
)

# -------------------------------------------------------
# 7) Column groups for the trainer (text / numeric / aux)
# -------------------------------------------------------
BASE_TEXT_COLS = DIALECT_TEXT_COLS + ["fanqie_upper_tok","fanqie_lower_tok"]
tone_prefixes = [tc[:4].lower() for tc in tone_cols]
NUMERIC_COLS = sorted([
    c for c in df_feat.columns
    if re.match(rf"^({'|'.join(map(re.escape, tone_prefixes))})_tone_(mean|max|len|has)$", c)
])
AUX_COLS = ["has_kun"]

# 8) Diagnostics (confirm Tang isn’t empty after normalization)
print("df_feat shape:", df_feat.shape)
print("Text cols:", len(BASE_TEXT_COLS), "| Numeric tone cols:", len(NUMERIC_COLS), "| Aux cols:", AUX_COLS)

coverage = {c: int((df_feat[c].astype(str).str.len() > 0).sum()) for c in DIALECT_TEXT_COLS}
print("Non-empty counts:", coverage)

df_feat.head(10)


df_feat shape: (68333, 47)
Text cols: 13 | Numeric tone cols: 32 | Aux cols: ['has_kun']
Non-empty counts: {'Old Chinese': 4966, 'Mandarin': 42603, 'Cantonese': 30844, 'Teochew': 8271, 'Hokkien': 6274, 'Hakka': 5072, 'Vietnamese': 8897, 'Japanese On': 14073, 'Korean (romanized)': 9884, 'Middle Chinese': 4965, 'Tang': 4466}


,char,Old Chinese,Mandarin,Cantonese,Teochew,Hokkien,Hakka,Vietnamese,Japanese On,Korean (romanized),...,midd_tone_len,midd_tone_has,viet_tone_mean,viet_tone_max,viet_tone_len,viet_tone_has,tang_tone_mean,tang_tone_max,tang_tone_len,tang_tone_has
0,%,,,,,pʰa˥,,,,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,の,,,,,e˨˦,,,,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,㐀,,qiū,jau1,,,,,,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,㐁,,tiàn,,,,,,,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,㐂,,,,,,,,,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,㐃,,,zim1,,,,,,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,㐄,,kuà,kwaa1,,,,,,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,㐅,,wǔ,ng5,,,,,,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,㐆,,yǐn,,,,,,,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,㐇,,,gau2,,,,,,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Index(['char', 'Old Chinese', 'Mandarin', 'Cantonese', 'Teochew', 'Hokkien',
       'Hakka', 'Wu', 'Gan', 'Xiang', 'Jin', 'Vietnamese', 'Japanese On',
       'Korean (romanized)', 'Middle Chinese', 'Old Chinese_is_pred',
       'Mandarin_is_pred', 'Cantonese_is_pred', 'Teochew_is_pred',
       'Hokkien_is_pred', 'Hakka_is_pred', 'Wu_is_pred', 'Gan_is_pred',
       'Xiang_is_pred', 'Jin_is_pred', 'Vietnamese_is_pred',
       'Japanese On_is_pred', 'Korean (romanized)_is_pred',
       'Middle Chinese_is_pred'],
      dtype='object')


In [48]:
# === Cell: Accuracy-first per-dialect training + gated imputation (abstain if unsure, but not forever) ===
import numpy as np
import pandas as pd
import gc, math, re
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neighbors import NearestNeighbors
from sklearn.utils.class_weight import compute_class_weight

# ------------------------
# 0) Dialects & groupings
# ------------------------
TARGET_LANGS = [
    'Old Chinese',
    'Mandarin','Cantonese','Teochew','Hokkien','Hakka',
    'Vietnamese','Japanese On','Korean (romanized)',
    'Middle Chinese','Tang'
]

# Curated counts (for threshold grouping)
CURATED_N = {
    'Old Chinese': 4966, 'Mandarin': 42603, 'Cantonese': 30844, 'Teochew': 8271,
    'Hokkien': 6274, 'Hakka': 5072, 'Vietnamese': 8897, 'Japanese On': 14073,
    'Korean (romanized)': 9884, 'Middle Chinese': 4965, 'Tang': 4466
}

def _group(d):
    n = CURATED_N.get(d, 0)
    if n >= 10000: return "large"
    if n >= 6000:  return "mid"
    return "small"

# Thresholds (slightly relaxed from your last run)
# Global thresholds (no per-dialect hacks)
THRESH = {
    "large": dict(p1=0.80, margin=0.04, entropy=1.70, knn_agree=0.40, knn_sim=0.68, min_support=4),
    "mid":   dict(p1=0.79, margin=0.04, entropy=1.70, knn_agree=0.40, knn_sim=0.68, min_support=3),
    "small": dict(p1=0.78, margin=0.03, entropy=1.75, knn_agree=0.38, knn_sim=0.66, min_support=3),
}

def get_thresh(dialect: str):
    return THRESH[_group(dialect)].copy()

# High-confidence override (p1-only), global by group
HIGH_OVERRIDE = {
    "large": 0.86,
    "mid":   0.85,
    "small": 0.84,
}
def get_high_override(dialect: str):
    return HIGH_OVERRIDE[_group(dialect)]



# Floors (not used unless you add an auto-relax pass later)
FLOOR = {
    "large": dict(p1=0.84, margin=0.10, entropy=1.35, knn_agree=0.50, knn_sim=0.75),
    "mid":   dict(p1=0.82, margin=0.08, entropy=1.40, knn_agree=0.50, knn_sim=0.75),
    "small": dict(p1=0.80, margin=0.06, entropy=1.45, knn_agree=0.50, knn_sim=0.75),
}

# Majority bump disabled (it was blocking near everything for OC/MC)
EXTRA_MAJ_BUMP = {}
MAJ_BUMP = dict(p1=0.02, knn_agree=0.05)

# Regex / phonotactics sanity (permissive; hyphen sits at end of class)
DIALECT_REGEX = {
    'Mandarin': re.compile(r"^[A-Za-z0-9\s'ːˑ˞\u0361ɑɐʯʰʲ̃ˈˌ˥˦˧˨˩-]+$"),
    'Cantonese': re.compile(r"^[A-Za-z0-9\s'ːˑ˞\u0361ɑɐʯʰʲ̃ˈˌ˥˦˧˨˩-]+$"),
    'Teochew': re.compile(r"^[A-Za-z0-9\s'ːˑ˞\u0361ɑɐʯʰʲ̃ˈˌ˥˦˧˨˩-]+$"),
    'Hokkien': re.compile(r"^[A-Za-z0-9\s'ːˑ˞\u0361ɑɐʯʰʲ̃ˈˌ˥˦˧˨˩-]+$"),
    'Hakka': re.compile(r"^[A-Za-z0-9\s'ːˑ˞\u0361ɑɐʯʰʲ̃ˈˌ˥˦˧˨˩-]+$"),
    'Vietnamese': re.compile(r"^[A-Za-z0-9\s'ːˑ˞\u0361ăâêôơưđĂÂÊÔƠƯĐ̣́̀̉̃˥˦˧˨˩-]+$"),
    'Japanese On': re.compile(r"^[A-Za-z0-9\s'ːˑ˞\u0361āīūēō-]+$"),
    'Korean (romanized)': re.compile(r"^[A-Za-z0-9\s'ːˑ˞\u0361-]+$"),
    'Old Chinese': re.compile(r"^[A-Za-z0-9\s'ːˑ˞\u0361ɑɐʯʰʲ̃ˈˌ˥˦˧˨˩\*\?-]+$"),
    'Middle Chinese': re.compile(r"^[A-Za-z0-9\s'ːˑ˞\u0361ɑɐʯʰʲ̃ˈˌ˥˦˧˨˩\*\?-]+$"),
    'Tang': re.compile(r"^[A-Za-z0-9\s'ːˑ˞\u0361ɑɐʯʰʲ̃ˈˌ˥˦˧˨˩\u4E00-\u9FFF\u3400-\u4DBF\uF900-\uFAFF-]+$"),
}

def _ok_by_regex(dialect, s):
    if not isinstance(s, str) or not s.strip():
        return False
    pat = DIALECT_REGEX.get(dialect)
    if pat is None:
        return True
    return bool(pat.match(s.strip()))

def _nonempty_mask(series):
    s = series.astype(str).str.strip()
    return s.ne("") & s.ne("nan")

# -------------------------
# 1) Build model components
# -------------------------
def _make_vectorizer(text_cols, numeric_cols):
    class _Concat:
        def fit(self, X, y=None): return self
        def transform(self, X):
            dfX = pd.DataFrame(X, columns=text_cols)
            return dfX.astype(str).apply(lambda r: " ␟ ".join([t for t in r if t]), axis=1).values

    vect = ColumnTransformer(
        transformers=[
            ("txt", Pipeline([
                ("concat", _Concat()),
                ("tfidf", TfidfVectorizer(
                    ngram_range=(1,2),
                    min_df=2,
                    max_df=0.98,
                    strip_accents=None,
                    analyzer="char_wb"
                ))
            ]), text_cols),
            ("num", Pipeline([
                ("scaler", StandardScaler(with_mean=False))
            ]), numeric_cols)
        ],
        remainder="drop",
        sparse_threshold=0.3
    )
    return vect

def _entropy(p):
    p = np.clip(p, 1e-12, 1.0)
    return float(-np.sum(p * np.log(p)))

def _majority_label_and_frac(y):
    if len(y) == 0: return None, 0.0
    vals, counts = np.unique(y, return_counts=True)
    idx = np.argmax(counts)
    return vals[idx], counts[idx] / len(y)

def train_one_dialect(df_feat, target_col, test_size=0.2, min_class_count=3, random_state=42):
    y_all = df_feat[target_col].astype(str).str.strip()
    mask = _nonempty_mask(y_all)
    if mask.sum() < 20:
        return None, None, None, None, None

    # Filter low-support labels
    vc = y_all[mask].value_counts()
    keep_labels = set(vc[vc >= min_class_count].index.tolist())
    keep_mask = mask & y_all.isin(keep_labels)
    if keep_mask.sum() < 20 or len(keep_labels) < 2:
        return None, None, None, None, None

    y = y_all[keep_mask]

    # Features: all base text except the target, plus numeric
    text_cols = [c for c in BASE_TEXT_COLS if c != target_col]
    num_cols  = NUMERIC_COLS + AUX_COLS
    X_text = df_feat.loc[keep_mask, text_cols]
    X_num  = df_feat.loc[keep_mask, num_cols]
    X      = pd.concat([X_text.reset_index(drop=True), X_num.reset_index(drop=True)], axis=1)

    # Train/valid split (ensure n_test >= n_classes for stratification)
    n_classes = y.nunique()
    n_rows = len(y)

    # Minimal test fraction so each class can appear at least once in the test set (+ small margin)
    min_frac = (n_classes + 1) / n_rows + 0.02
    ts = max(test_size, min_frac)
    ts = min(ts, 0.50)  # cap at 50% so train set isn't too tiny

    try:
        X_train, X_valid, y_train, y_valid = train_test_split(
            X, y, test_size=ts, random_state=random_state, stratify=y
        )
    except ValueError:
        # If still impossible (ultra many classes vs. rows), drop stratification gracefully
        X_train, X_valid, y_train, y_valid = train_test_split(
            X, y, test_size=ts, random_state=random_state, stratify=None
        )


    # Class weights to reduce majority collapse
    classes = np.unique(y_train)
    cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weight = {cls: w for cls, w in zip(classes, cw)}

    vect = _make_vectorizer(text_cols, num_cols)
    base = LogisticRegression(
        solver="saga", penalty="l2", class_weight=class_weight,
        max_iter=2000, n_jobs=-1, verbose=0
    )

    pipe = Pipeline([("vec", vect), ("lr", base)])
    pipe.fit(X_train, y_train)

    # Calibrate (prefer isotonic; fallback to sigmoid)
    try:
        calibrated = CalibratedClassifierCV(pipe, cv="prefit", method="isotonic")
        calibrated.fit(X_valid, y_valid)
    except Exception:
        calibrated = CalibratedClassifierCV(pipe, cv="prefit", method="sigmoid")
        calibrated.fit(X_valid, y_valid)

    # Neighbor index on TRAIN
    vec_only = vect.fit(X_train, y_train)   # embed TRAIN space
    X_train_emb = vec_only.transform(X_train)
    nn = NearestNeighbors(n_neighbors=min(20, X_train_emb.shape[0]), metric="cosine")
    nn.fit(X_train_emb)

    maj_label, maj_frac = _majority_label_and_frac(y_train.values)

    # Valid metrics
    proba_v = calibrated.predict_proba(X_valid)
    pred_v  = calibrated.classes_[np.argmax(proba_v, axis=1)]
    top1 = accuracy_score(y_valid, pred_v)
    try:
        ll = log_loss(y_valid, proba_v, labels=calibrated.classes_)
    except Exception:
        ll = None

    report = dict(rows=len(y_valid), classes=len(calibrated.classes_), top1=round(float(top1),3),
                  logloss=float(ll) if ll is not None else None)

    return calibrated, vec_only, nn, calibrated.classes_, report, (maj_label, maj_frac), (X_train, y_train)

def _gate_one(dialect, proba_row, classes, neigh_labels, neigh_sims, thresholds, maj_info):
    p_sorted_idx = np.argsort(-proba_row)
    p1 = float(proba_row[p_sorted_idx[0]])
    p2 = float(proba_row[p_sorted_idx[1]]) if len(p_sorted_idx) > 1 else 0.0
    margin = p1 - p2
    ent = _entropy(proba_row)
    pred_label = classes[p_sorted_idx[0]]

    # kNN agree/sim
    if len(neigh_labels) > 0:
        agree = float(np.mean(np.array(neigh_labels) == pred_label))
        mean_sim = float(np.mean(neigh_sims))
    else:
        agree, mean_sim = 0.0, 0.0

    # Optional OC/MC bump (disabled by default)
    thr = thresholds.copy()
    maj_label, _ = maj_info
    if dialect in EXTRA_MAJ_BUMP and maj_label is not None and pred_label == maj_label:
        thr["p1"]  += MAJ_BUMP["p1"]
        thr["knn_agree"] += MAJ_BUMP["knn_agree"]

    # High-confidence override (p1-only), still passes regex later
    if p1 >= get_high_override(dialect):
        return pred_label

    # Otherwise require all gates
    if not (p1 >= thr["p1"] and margin >= thr["margin"] and ent <= thr["entropy"]
            and agree >= thr["knn_agree"] and mean_sim >= thr["knn_sim"]):
        return None

    return pred_label

def _predict_missing_with_gates(df_feat, model_pack, target_col, group, min_nonempty_sources=2, batch_size=4000):
    clf, vec_only, nn, classes, report, maj_info, train_pack = model_pack
    if clf is None:
        print(f"↪ Skipping {target_col}: no model")
        return 0, 0

    text_cols = [c for c in BASE_TEXT_COLS if c != target_col]
    num_cols  = NUMERIC_COLS + AUX_COLS

    # Mask of rows to try (target empty + enough other evidence)
    tgt_empty = df_feat[target_col].astype(str).str.strip().eq("")
    other_text = df_feat[text_cols].astype(str)
    # vectorized count of non-empty sources (no deprecated applymap)
    nonempty_sources = other_text.replace(r'^\s*$', np.nan, regex=True).notna().sum(axis=1)

    mask = tgt_empty & nonempty_sources.ge(min_nonempty_sources)
    idxs = df_feat.index[mask].to_list()
    if not idxs:
        print(f"↪ Nothing to impute for '{target_col}'.")
        return 0, 0

    thr = get_thresh(target_col)

    # Label support (rare-label guard)
    _, y_train = train_pack
    label_support = pd.Series(y_train).value_counts()

    accepted = 0
    attempted = 0

    for start in tqdm(range(0, len(idxs), batch_size), desc=f"Predicting {target_col}", unit="char", leave=False, mininterval=2.0):
        batch_idx = idxs[start:start+batch_size]
        Xb_text = df_feat.loc[batch_idx, text_cols]
        Xb_num  = df_feat.loc[batch_idx, num_cols]
        Xb      = pd.concat([Xb_text.reset_index(drop=True), Xb_num.reset_index(drop=True)], axis=1)

        proba = clf.predict_proba(Xb)

        # kNN agreement for batch
        Xb_emb = vec_only.transform(Xb)
        n_k = min(20, nn.n_neighbors)
        dists, nbrs = nn.kneighbors(Xb_emb, n_neighbors=n_k)
        sims = 1.0 - dists  # cosine similarity → similarity

        for i, row_idx in enumerate(batch_idx):
            attempted += 1
            pr = proba[i]
            nbr_ids = nbrs[i]
            nbr_labels = train_pack[1].iloc[nbr_ids].values
            nbr_sims = sims[i]

            pred = _gate_one(target_col, pr, classes, nbr_labels, nbr_sims, thr, maj_info)

            # Rare-label extra requirement
            if pred is not None and label_support.get(pred, 0) < get_thresh(target_col)["min_support"]:
                # demand stronger p1 for rare labels
                if float(np.max(pr)) < (HIGH_OVERRIDE[_group(target_col)] + 0.02):
                    pred = None

            # Regex sanity at the end
            if pred is not None and not _ok_by_regex(target_col, pred):
                pred = None

            if pred is None:
                continue

            # Write back accepted prediction
            df_feat.at[row_idx, target_col] = pred
            flag_col = f"{target_col}_is_pred"
            if flag_col not in df_feat.columns:
                df_feat[flag_col] = 0
            df_feat.at[row_idx, flag_col] = 1
            accepted += 1

        # cleanup
        del Xb, Xb_text, Xb_num, Xb_emb, proba, nbrs, dists, sims
        gc.collect()

    return accepted, attempted

# ----------------------------------------
# 2) Train, evaluate, and gated imputation
# ----------------------------------------
models = {}
reports = []
impute_log = []

print("\n=== Accuracy-first training + gated imputation (abstain if unsure, but allow high-confidence overrides) ===\n")

for tgt in tqdm(TARGET_LANGS, desc="Dialects", unit="dial"):
    print(f"\n{'='*72}\nTraining target: {tgt} (group: {_group(tgt)})\n{'='*72}")

    test_size = 0.25 if _group(tgt) == "small" else 0.20
    min_class = THRESH[_group(tgt)]["min_support"]

    pack = train_one_dialect(
        df_feat, target_col=tgt,
        test_size=test_size, min_class_count=min_class, random_state=42
    )
    if pack[0] is None:
        print(f"⚠️ '{tgt}': insufficient data or classes after filtering — skipping.")
        reports.append({"target": tgt, "rows": 0, "classes": 0, "top1": None, "logloss": None})
        continue

    clf, vec_only, nn, classes, rep, maj_info, train_pack = pack
    models[tgt] = pack
    rep = {"target": tgt, **(rep or {})}
    print("Valid metrics:", rep)
    reports.append(rep)

    # Impute (lighter evidence for "large": 1 source; others: 2)
    min_src = 1 if _group(tgt) == "large" else 2
    accepted, attempted = _predict_missing_with_gates(
        df_feat, pack, tgt, _group(tgt),
        min_nonempty_sources=min_src,
        batch_size=4000
    )

    print(f"🧩 {tgt}: accepted {accepted:,} / attempted {attempted:,} predictions (abstained {attempted-accepted:,}).")
    if accepted:
        impute_log.append((tgt, accepted, attempted))

# Summary
summ_df = pd.DataFrame(reports).sort_values(by=["rows","top1"], ascending=[False, False]).reset_index(drop=True)
print("\n=== Validation Summary ===")
display(summ_df)

if impute_log:
    filled_df = pd.DataFrame(impute_log, columns=["target", "accepted", "attempted"]).sort_values("accepted", ascending=False)
    print("\n=== Imputation (accepted only) ===")
    display(filled_df)
else:
    print("\n(No accepted predictions — abstained everywhere.)")



=== Accuracy-first training + gated imputation (abstain if unsure, but allow high-confidence overrides) ===



Dialects:   0%|                                                                               | 0/11 [00:00<?, ?dial/s]


Training target: Old Chinese (group: small)


C:\Users\zacha\miniconda3\envs\env1\Lib\site-packages\sklearn\utils\multiclass.py:213: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  y_type = type_of_target(y, input_name="y")
C:\Users\zacha\miniconda3\envs\env1\Lib\site-packages\sklearn\calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(
C:\Users\zacha\miniconda3\envs\env1\Lib\site-packages\sklearn\preprocessing\_label.py:528: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  y_type = type_of_target(y)
C:\Users\zacha\miniconda3\envs\env1\Lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number

Valid metrics: {'target': 'Old Chinese', 'rows': 309, 'classes': 287, 'top1': 0.595, 'logloss': 1.7811596932809595}



Dialects:   9%|██████▍                                                                | 1/11 [00:14<02:26, 14.63s/dial]

... (40 outputs left out)

🧩 Tang: accepted 0 / attempted 24,619 predictions (abstained 24,619).

=== Validation Summary ===


,target,rows,classes,top1,logloss
0,Mandarin,8413,1183,0.344,4.175326
1,Cantonese,5998,1381,0.558,2.413808
2,Japanese On,2033,587,0.586,1.968354
3,Korean (romanized),1849,511,0.608,1.817668
4,Teochew,1441,908,0.557,2.100058
5,Vietnamese,1169,1084,0.312,4.346580
6,Hakka,1042,703,0.673,1.607057
7,Hokkien,970,850,0.681,1.410026
8,Middle Chinese,693,640,0.596,2.074998
9,Tang,397,369,0.736,1.311967



=== Imputation (accepted only) ===


,target,accepted,attempted
0,Cantonese,1212,20349
4,Korean (romanized),628,19389
3,Japanese On,248,37037
1,Teochew,15,20730
2,Hokkien,1,22684
5,Middle Chinese,1,24130


In [50]:
# === Merge model imputations back into original data, excluding JP/KR, and tag [ML] ===
import pandas as pd

EXCLUDE_DIALECTS = {"Japanese On", "Korean (romanized)"}

# 1) Start from the original table
df_imputed = df_raw.copy().reset_index(drop=True)
df_feat = df_feat.reset_index(drop=True)

# 2) For each dialect, fill in predictions (except excluded), tag with [ML], don't overwrite originals
inserted_counts = {}
for col in TARGET_LANGS:
    if col in EXCLUDE_DIALECTS:
        continue
    flag = f"{col}_is_pred"
    if flag not in df_feat.columns:
        continue

    # predicted rows according to the ML pipeline
    pred_mask = (df_feat[flag] == 1)

    # only fill where the original df_raw cell is truly empty
    orig_empty = df_imputed[col].astype(str).str.strip().replace({"nan": ""}) == ""
    to_fill = pred_mask & orig_empty

    # append [ML] with no space per your spec
    df_imputed.loc[to_fill, col] = df_feat.loc[to_fill, col].astype(str) + "[ML]"
    inserted_counts[col] = int(to_fill.sum())

# 3) Save to CSV
out_path = "combined_dialects_imputed_tagged_excl_jp_kr.csv"
df_imputed.to_csv(out_path, index=False)

# 4) Quick summary
print("✅ Inserted [ML] predictions (excluding Japanese On & Korean (romanized)):")
for k, v in inserted_counts.items():
    print(f"  - {k}: {v}")
print(f"\n💾 Exported to: {out_path}")



✅ Inserted [ML] predictions (excluding Japanese On & Korean (romanized)):
  - Cantonese: 1212
  - Teochew: 21
  - Hokkien: 62
  - Middle Chinese: 3
  - Tang: 13

💾 Exported to: combined_dialects_imputed_tagged_excl_jp_kr.csv
